# Enkrypt AI Red Team Testing - Tool Based Indirect Prompt Injection Attacks

## Simplified Configuration for Tool Based Indirect Prompt Injection Testing

This notebook provides a **simplified setup** for red team testing indirect prompt injection attacks:
- **Webpage Injection Test**: Test type for indirect injection attacks (system description and policy handled automatically)
- **Indirect Injection Format**: All attacks are hidden in tool output chunks (HTML, documents, code blocks, metadata) rather than direct user queries

## 🎯 Test Type Available

The **`webpage_injection_nt_test`** test type simplifies the testing process:

- **No Configuration Required**: You don't need to set system description or policy description - we handle that in the backend
- **Pre-configured Attacks**: The test automatically generates appropriate indirect injection attacks
- **Ready to Use**: Simply configure your model endpoint and submit the test

## 📋 Prerequisites

- ✅ Enkrypt AI API key is configured
- ✅ A model is pre-saved in your Enkrypt AI account

## 🎯 How It Works

1. **Setup**: Load API key and configure connection
2. **Configure Dataset Parameters**: Set scenarios, categories, and depth for test generation
3. **Build Test Configuration**: Enable Webpage Injection Test
4. **Submit**: Send your test to Enkrypt AI for execution


## 1. Setup and Connection


In [ ]:
import requests
import json
import os
from dotenv import load_dotenv
import time

load_dotenv()

# Connection Configuration
ENKRYPT_API_KEY = os.getenv('ENKRYPTAI_API_KEY')

# Override API key if provided
ENKRYPT_API_KEY = "QrostlU1SUi7fmQGigLQTQ3MmW971U88"

if not ENKRYPT_API_KEY:
    print("❌ ENKRYPTAI_API_KEY not found in environment variables")
    print("Please set your API key in the .env file")
else:
    print("✅ Enkrypt AI API key loaded")


## 2. Configure Dataset Parameters

Set scenarios, categories, and depth for test generation. It is best to start with values of 1, 1, 1.



In [ ]:
dataset_configuration = {
    "system_description": "test",
    "policy_description": None,
    "tools": None,
    "scenarios": 2,
    "categories": 2,
    "depth": 2,
    "max_prompts": 100,
    "risk_categories": None
}

print("✅ Configuration loaded!")
print(f"\n💡 Dataset parameters configured:")
print(f"   • Scenarios: {dataset_configuration['scenarios']}")
print(f"   • Categories: {dataset_configuration['categories']}")
print(f"   • Depth: {dataset_configuration['depth']}")
print(f"   • Max Prompts: {dataset_configuration['max_prompts']}")


## 3. Build Test Configuration

This section automatically builds the test configuration with:
- **Webpage Injection Test** enabled
- **Basic attack method** enabled
- **Sample percentage**


In [ ]:
# Build test configuration - Webpage Injection Test enabled
SAMPLE_PERCENTAGE = 100

redteam_test_configurations = {
    "webpage_injection_nt_test": {
        "sample_percentage": SAMPLE_PERCENTAGE,
        "attack_methods": {
            "basic": { "basic": { "params": {} } }
        }
    }
}

print("✅ Test configuration built successfully!")
print(f"📋 Test Type: webpage_injection_nt_test")
print(f"📊 Sample Percentage: {SAMPLE_PERCENTAGE}%")
print(f"🎯 Attack Method: Basic")


## 4. Create Red Team Task

Submit your configured test to Enkrypt AI for red team testing.

**Note**: Make sure to update the model name and version in the headers below to match your saved model.


In [ ]:
print("=== Creating Red Team Task ===")

# Generate a unique test name with timestamp
test_name = f"Tool_Based_Indirect_Injection_Redteam_Test_{int(time.time())}"
print(f"🎯 Test Name: {test_name}")

# Configure red team test payload
payload = {
    "test_name": test_name,
    "redteam_test_configurations": redteam_test_configurations,
    "dataset_configuration": dataset_configuration
}

print(json.dumps(payload, indent=2))

print("\n📋 Payload Preview:")
print(f"Test types included: {list(redteam_test_configurations.keys())}")
print(f"Attack method: Basic")


## Set Endpoint

In [ ]:

# Submit red team task using pre-saved model
# ⚠️ UPDATE THESE VALUES TO MATCH YOUR SAVED MODEL
url = "https://api.enkryptai.com/redteam/v3/model/add-custom-task"
headers = {
    "X-Enkrypt-Model": "GPT 5.2",  # Update this
    "X-Enkrypt-Model-Version": "1",  # Update this
    "apikey": ENKRYPT_API_KEY,
    "Content-Type": "application/json"
}

print("✅ Endpoint set")
print(f"Model: {headers['X-Enkrypt-Model']}")
print(f"Model Version: {headers['X-Enkrypt-Model-Version']}")


In [ ]:

print("\n🚀 Submitting red team task...")
try:
    response = requests.post(url, json=payload, headers=headers, timeout=60)
    
    if response.status_code == 200:
        result = response.json()
        print("✅ Red team task created successfully!")
        print(f"Response: {json.dumps(result, indent=2)}")
        
        print(f"\n📋 Test Configuration Summary:")
        print(f"   • Test Name: {test_name}")
        print(f"   • Test Type: webpage_injection_nt_test")
        print(f"   • Sample Rate: {SAMPLE_PERCENTAGE}%")
        print(f"   • Attack Method: Basic")
        
    else:
        print(f"⚠️ Task creation returned status {response.status_code}")
        try:
            error_details = response.json()
            print(f"Error details: {json.dumps(error_details, indent=2)}")
        except:
            print(f"Raw response: {response.text}")
            
except Exception as e:
    print(f"❌ Task creation failed: {e}")


## 🔗 Next Steps & Monitoring

### 🕐 Red Team Testing Timeline

Red team testing typically takes **30-120 minutes** depending on:
- Dataset size and complexity
- Model response times
- Attack method complexity

### 📊 Monitor Your Test Progress

Visit the Enkrypt AI dashboard to monitor your test in real-time:

```
🌐 Dashboard: https://app.enkryptai.com/redteam
```

**Dashboard Features:**
- ✅ Real-time progress tracking
- 📈 Live results as they come in
- 🔍 Detailed attack breakdowns
- 📊 Security score calculations


After the test is done, you can download the results with this code:

In [ ]:
import zipfile

print("=== Downloading Red Team Results ===")

# Use the test name from the task creation
# Make sure the test has completed before running this cell
url = "https://api.enkryptai.com/redteam/download-link"

TEST_NAME = "ENTER TEST NAME HERE"

headers = {
    "X-Enkrypt-Test-Name": TEST_NAME,
    "apikey": ENKRYPT_API_KEY
}

print(f"📥 Requesting download link for test: {test_name}")

try:
    response = requests.get(url, headers=headers)
    result = response.json()
    
    if 'link' in result:
        download_url = result['link']
        print(f"\n✅ Download link received")
        print(f"📦 Downloading file from: {download_url}")
        
        # Download the file
        download_response = requests.get(download_url)
        
        if download_response.status_code == 200:
            # Save the file
            filename = f"{test_name.replace(' ', '_')}_redteam_results.zip"
            with open(filename, 'wb') as f:
                f.write(download_response.content)
            print(f"✅ File downloaded successfully: {filename}")
            
            # Extract the zip file
            extract_folder = f"{test_name.replace(' ', '_')}_redteam_results"
            print(f"\n📂 Extracting to: {extract_folder}/")
            
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall(extract_folder)
            
            print(f"✅ File extracted successfully to: {extract_folder}/")
            
            # List extracted files
            extracted_files = os.listdir(extract_folder)
            print(f"\n📋 Extracted {len(extracted_files)} file(s):")
            for file in extracted_files[:10]:  # Show first 10 files
                print(f"   • {file}")
            if len(extracted_files) > 10:
                print(f"   ... and {len(extracted_files) - 10} more")
        else:
            print(f"❌ Failed to download file. Status code: {download_response.status_code}")
    else:
        print("❌ No download link found in response")
        print(f"Response: {json.dumps(result, indent=2)}")
        print("\n💡 Make sure the test has completed before downloading results.")
        
except Exception as e:
    print(f"❌ Error downloading results: {e}")
    print("\n💡 Make sure:")
    print("   • The test has completed")
    print("   • The test name is correct")
    print("   • Your API key is valid")